In [3]:
import pandas as pd
import numpy as np

In [23]:
import re

STATE_ZIP3_RANGES = {
    "Alabama": ((350, 369),), "Alaska": ((995, 999),), "Arizona": ((850, 865),),
    "Arkansas": ((716, 729),), "California": ((900, 961),), "Colorado": ((800, 816),),
    "Connecticut": ((60, 69),), "Delaware": ((197, 199),), "District of Columbia": ((200, 200),),
    "Florida": ((320, 349),), "Georgia": ((300, 319), (398, 398)), "Hawaii": ((967, 968),),
    "Idaho": ((832, 838),), "Illinois": ((600, 629),), "Indiana": ((460, 479),),
    "Iowa": ((500, 528),), "Kansas": ((660, 679),), "Kentucky": ((400, 427),),
    "Louisiana": ((700, 714),), "Maine": ((39, 49),), "Maryland": ((206, 219),),
    "Massachusetts": ((10, 27),), "Michigan": ((480, 499),), "Minnesota": ((550, 567),),
    "Mississippi": ((386, 397),), "Missouri": ((630, 658),), "Montana": ((590, 599),),
    "Nebraska": ((680, 693),), "Nevada": ((889, 898),), "New Hampshire": ((30, 38),),
    "New Jersey": ((70, 89),), "New Mexico": ((870, 884),), "New York": ((100, 149),),
    "North Carolina": ((270, 289),), "North Dakota": ((580, 588),), "Ohio": ((430, 459),),
    "Oklahoma": ((730, 731), (734, 749)), "Oregon": ((970, 979),),
    "Pennsylvania": ((150, 196),), "Rhode Island": ((28, 29),),
    "South Carolina": ((290, 299),), "South Dakota": ((570, 577),),
    "Tennessee": ((370, 385),), "Texas": ((733, 733), (750, 799), (885, 885)),
    "Utah": ((840, 847),), "Vermont": ((50, 59),),
    "Virginia": ((201, 201), (220, 246)), "Washington": ((980, 994),),
    "West Virginia": ((247, 268),), "Wisconsin": ((530, 549),), "Wyoming": ((820, 831),),
}
STATE_PATTERN = re.compile(
    r"\b(?P<state>" + "|".join(
        re.escape(state) for state in sorted(STATE_ZIP3_RANGES, key=len, reverse=True)
    ) + r")\b",
    re.IGNORECASE,
)
US_SUFFIX = re.compile(
    r"\bUnited States(?: of America)?\b(?:\s*\([^)]*\))?\s*$",
    re.IGNORECASE,
)
ZIP_PATTERN = re.compile(r"\b(?P<zip>\d{5})(?:-\d{4})?\b")
STATE_ZIP_ABBREVIATION_PATTERN = re.compile(
    r",\s*(?P<abbreviation>[A-Z]{2})\s+(?=\d{5}(?:-\d{4})?\b)"
)
STATE_ABBREVIATIONS = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas",
    "CA": "California", "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware",
    "DC": "District of Columbia", "FL": "Florida", "GA": "Georgia", "HI": "Hawaii",
    "ID": "Idaho", "IL": "Illinois", "IN": "Indiana", "IA": "Iowa", "KS": "Kansas",
    "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi",
    "MO": "Missouri", "MT": "Montana", "NE": "Nebraska", "NV": "Nevada",
    "NH": "New Hampshire", "NJ": "New Jersey", "NM": "New Mexico", "NY": "New York",
    "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio", "OK": "Oklahoma",
    "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah",
    "VT": "Vermont", "VA": "Virginia", "WA": "Washington", "WV": "West Virginia",
    "WI": "Wisconsin", "WY": "Wyoming",
}


def get_state(address):
    state_match = STATE_PATTERN.search(address)
    if state_match:
        return state_match.group("state").title()

    abbreviation_match = STATE_ZIP_ABBREVIATION_PATTERN.search(address)
    if abbreviation_match:
        state = STATE_ABBREVIATIONS.get(abbreviation_match.group("abbreviation"))
        if state:
            return state

    if US_SUFFIX.search(address):
        zip_match = ZIP_PATTERN.search(address)
        if zip_match:
            zip3 = int(zip_match.group("zip")[:3])
            for state, ranges in STATE_ZIP3_RANGES.items():
                if any(start <= zip3 <= end for start, end in ranges):
                    return state

    return pd.NA


df = pd.read_csv("filtered.csv", low_memory=False)
df = df.dropna(subset=["address"]).copy()
df["state"] = df["address"].map(get_state)

print(f"Rows retained after removing null addresses: {len(df):,}")
print(f"Addresses with no identified U.S. state: {df['state'].isna().sum():,}")

ACTIVITY_PATTERNS = {
    "restaurant_dining": r"\b(?:restaurant|chinese|cafe|vegan|coffee shop|bakery|diner|bistro|eatery|food|dining)\b",
    "cooking_experience": r"\b(?:cooking|culinary) (?:class|experience|workshop|lesson|demonstration)\b|\bcooking class\b",
    "live_music": r"\b(?:live music|concerts?|music venue|live performance|jazz club|orchestra)\b",
    "sightseeing": r"\b(?:tourist attraction|tourist_attraction|landmark|point_of_interest|sightseeing|observation deck|viewpoint|botanical garden|garden|park|monument|memorial|museum|historic site|guided tour)\b",
    "running": r"\b(?:running|run club|running trails?|running track|marathon)\b",
    "water_sports": r"\b(?:water sports?|beach|lake|boating|boat tours?|kayaking|canoeing|surfing|snorkeling|scuba diving|paddle ?boarding|jet skiing|swimming|water park)\b",
    "walking": r"\b(?:walking|walkway|walking tour|walking trails?|promenade|footpath|walking path)\b",
    "nightlife": r"\b(?:bar|pub|night ?club|nightlife|cocktail lounge|bars & clubs|brewery|wine bar)\b",
    "hiking": r"\b(?:hiking|hike|mountain|hiking trails?|trekking|mountain trails?)\b",
    "photography": r"\b(?:photography|photograph|photo studio|photo tour|photo spot|photographic)\b",
    "zoo": r"\b(?:zoo|aquarium|zoological park|)\b",
    "shopping": r"\b(?:shopping|clothing|clothing_store|mall|store|retail)\b",
    "art": r"\b(?:art|movie|theatre|movie_theatre|museum|gallery|culture|cultural)\b",
    "history": r"\b(?:history|palace|civil|castle|historic|museum|gallery|culture|culturl)\b",
    "spa": r"\b(?:spa|wellness|massage)\b",
    "beauty": r"\b(?:beauty|cosmetics|skincare|salon)\b",
    "reading": r"\b(?:reading|book|library|book_store)\b",
    "hotel": r"\b(?:hotel|resort|inn|motel|bnb|hostel)\b",
}

ACTIVITY_PATTERNS = {
    activity: re.compile(pattern, re.IGNORECASE)
    for activity, pattern in ACTIVITY_PATTERNS.items()
}


ACTIVITY_TEXT_COLUMNS = ["name", "description", "category", "reviews", "combined_field"]


def get_activities(row):
    activity_text = " ".join(
        str(row[column])
        for column in ACTIVITY_TEXT_COLUMNS
        if column in row.index and pd.notna(row[column])
    ).replace("_", " ")
    return [
        activity
        for activity, pattern in ACTIVITY_PATTERNS.items()
        if pattern.search(activity_text)
    ]


df["activity"] = df.apply(get_activities, axis=1)
print(f"Rows without an activity match: {df['activity'].map(len).eq(0).sum():,}")

Rows retained after removing null addresses: 18,168
Addresses with no identified U.S. state: 2,931
Rows without an activity match: 0


In [16]:
# df.to_csv("activities_updated.csv", index=False)

In [24]:
df = df.dropna(subset=["state"])

In [25]:
df["activity"].value_counts(dropna=False)

activity
[zoo]                                                              7745
[sightseeing, zoo]                                                 1377
[water_sports, zoo]                                                1097
[sightseeing, zoo, art, history]                                   1068
[zoo, hotel]                                                        366
                                                                   ... 
[sightseeing, nightlife, hiking, zoo, art, history]                   1
[restaurant_dining, water_sports, nightlife, zoo, art, history]       1
[restaurant_dining, water_sports, zoo, hotel]                         1
[water_sports, nightlife, zoo, art, hotel]                            1
[restaurant_dining, zoo, art, hotel]                                  1
Name: count, Length: 322, dtype: int64

In [26]:
df.to_csv("activities_updated.csv", index=False)